# 02 — RFM Customer Segmentation
**Project:** Olist Brazilian E-Commerce — Customer Behavior & Cohort Analysis

RFM (Recency · Frequency · Monetary) is the gold-standard framework for customer segmentation in e-commerce. Each customer receives a 1–5 score on each dimension, enabling 8 actionable segments with clear marketing implications.

| Dimension | Definition | Scoring |
|-----------|-----------|--------|
| **Recency** | Days since last purchase | 5 = most recent, 1 = oldest |
| **Frequency** | Total number of orders | 5 = most frequent |
| **Monetary** | Total spend (R$) | 5 = highest spend |

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_raw, build_master
from src.rfm import compute_rfm, score_rfm, segment_summary
from src.utils import apply_style, SEGMENT_COLORS

apply_style()
raw = load_raw()
master = build_master(raw)
print('Data loaded ✓')

## 2.1 Compute Raw RFM Metrics

In [ ]:
rfm = compute_rfm(master)
print(f'Customers in RFM table: {len(rfm):,}')
rfm[['recency', 'frequency', 'monetary']].describe().round(2)

## 2.2 Score & Segment Customers

In [ ]:
rfm = score_rfm(rfm)
print('Segment distribution:')
print(rfm['segment'].value_counts().to_string())

## 2.3 Segment Summary — Revenue & Customer Counts

In [ ]:
seg_sum = segment_summary(rfm)
seg_sum.style.format({
    'avg_recency': '{:.0f}d',
    'avg_frequency': '{:.2f}',
    'avg_monetary': 'R$ {:.0f}',
    'total_revenue': 'R$ {:,.0f}',
    'revenue_pct': '{:.1f}%'
}).background_gradient(subset=['total_revenue'], cmap='Greens')

## 2.4 Segment Distribution — Donut & Revenue Bar

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = [SEGMENT_COLORS.get(s, '#ccc') for s in seg_sum['segment']]

# Donut
wedges, texts, autotexts = axes[0].pie(
    seg_sum['customers'], labels=None,
    autopct=lambda p: f'{p:.1f}%' if p > 4 else '',
    colors=colors, startangle=140, pctdistance=0.80,
    wedgeprops={'width': 0.55, 'edgecolor': 'white', 'linewidth': 2}
)
axes[0].legend(wedges, seg_sum['segment'], title='Segment', loc='center left',
               bbox_to_anchor=(1.0, 0.5), fontsize=9)
axes[0].set_title('Customer Count by Segment')

# Revenue bar
bars = axes[1].barh(seg_sum['segment'], seg_sum['total_revenue']/1000,
                    color=colors, edgecolor='white')
for bar, val in zip(bars, seg_sum['total_revenue']):
    axes[1].text(bar.get_width()+20, bar.get_y()+bar.get_height()/2,
                 f'R$ {val/1000:.0f}K', va='center', fontsize=9)
axes[1].set_xlabel('Total Revenue (R$ thousands)')
axes[1].set_title('Revenue by Segment')
axes[1].invert_yaxis()

fig.suptitle('RFM Customer Segmentation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 2.5 RFM Scatter — Recency vs Monetary

In [ ]:
sample = rfm.sample(min(8000, len(rfm)), random_state=42)

fig, ax = plt.subplots(figsize=(11, 7))
for seg, grp in sample.groupby('segment'):
    ax.scatter(grp['recency'], grp['monetary'], c=SEGMENT_COLORS.get(seg, '#ccc'),
               label=seg, alpha=0.55, s=grp['frequency']*30+10,
               edgecolors='white', linewidths=0.4)
ax.set_xlabel('Recency (days since last order)')
ax.set_ylabel('Monetary Value (R$)')
ax.set_title('RFM Scatter — Recency vs Monetary\n(bubble size = Frequency)')
ax.legend(loc='upper right', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
plt.tight_layout()
plt.show()

## 2.6 RFM Heatmap — Score Distributions

In [ ]:
pivot = rfm.groupby(['r_score', 'f_score'])['monetary'].mean().unstack()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot.astype(float), annot=True, fmt='.0f', cmap='YlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Avg Monetary (R$)'})
ax.set_title('Average Monetary Value by R-Score × F-Score\n(Higher = Better)')
ax.set_xlabel('Frequency Score')
ax.set_ylabel('Recency Score')
plt.tight_layout()
plt.show()

### Key Takeaways
- **At-Risk** is the largest segment (24%) and represents R$ 3.7M — the highest-priority reactivation target
- **Champions + Loyal Customers** drive 37% of revenue from only 36% of customers
- **Can't Lose Them** (8,671 customers) have high monetary value but very low recency — urgent win-back needed

---
**Next:** [03_cohort_analysis.ipynb](03_cohort_analysis.ipynb)